In [1]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = "/content/drive/MyDrive/pgm-final"

Mounted at /content/drive


In [2]:
import numpy as np

features_L1 = np.load(f"{PROJECT_DIR}/activations/features_L1_pos39.npy")
features_L2 = np.load(f"{PROJECT_DIR}/activations/features_L2_pos39.npy")

print("L1 features shape:", features_L1.shape)
print("L2 features shape:", features_L2.shape)
print("L1 dtype:", features_L1.dtype)
print("L2 memory:", features_L2.nbytes / 1e6, "MB")

L1 features shape: (5000, 24576)
L2 features shape: (5000, 24576)
L1 dtype: float32
L2 memory: 491.52 MB


In [3]:
firing_rate_L1 = (features_L1 > 0).mean(axis=0)
firing_rate_L2 = (features_L2 > 0).mean(axis=0)

print("L1 firing rates shape:", firing_rate_L1.shape)
print("L2 firing rates shape:", firing_rate_L2.shape)

print("\nL1 firing rate stats:")
print(f"  min:    {firing_rate_L1.min():.4f}")
print(f"  median: {np.median(firing_rate_L1):.4f}")
print(f"  max:    {firing_rate_L1.max():.4f}")
print(f"  mean:   {firing_rate_L1.mean():.4f}")

print("\nL2 firing rate stats:")
print(f"  min:    {firing_rate_L2.min():.4f}")
print(f"  median: {np.median(firing_rate_L2):.4f}")
print(f"  max:    {firing_rate_L2.max():.4f}")
print(f"  mean:   {firing_rate_L2.mean():.4f}")

L1 firing rates shape: (24576,)
L2 firing rates shape: (24576,)

L1 firing rate stats:
  min:    0.0000
  median: 0.0002
  max:    1.0000
  mean:   0.0009

L2 firing rate stats:
  min:    0.0000
  median: 0.0002
  max:    0.9984
  mean:   0.0012


In [4]:
LOWER = 0.05
UPPER = 0.95

mask_L1 = (firing_rate_L1 >= LOWER) & (firing_rate_L1 <= UPPER)
mask_L2 = (firing_rate_L2 >= LOWER) & (firing_rate_L2 <= UPPER)

print(f"L1: {mask_L1.sum()} features pass filter")
print(f"L2: {mask_L2.sum()} features pass filter")
print(f"Total variables for structure learning: {mask_L1.sum() + mask_L2.sum()}")

L1: 30 features pass filter
L2: 46 features pass filter
Total variables for structure learning: 76


In [5]:
indices_L1 = np.where(mask_L1)[0]
indices_L2 = np.where(mask_L2)[0]

print("First 5 L1 feature IDs that passed:", indices_L1[:5])
print("First 5 L2 feature IDs that passed:", indices_L2[:5])
print("Number of L1:", len(indices_L1), " Number of L2:", len(indices_L2))

First 5 L1 feature IDs that passed: [ 482  567 1153 1230 2064]
First 5 L2 feature IDs that passed: [ 878 1620 2325 2632 3545]
Number of L1: 30  Number of L2: 46


In [6]:
data_L1 = features_L1[:, indices_L1]
data_L2 = features_L2[:, indices_L2]

print("L1 subset shape:", data_L1.shape)
print("L2 subset shape:", data_L2.shape)

X_continuous = np.concatenate([data_L1, data_L2], axis=1)
print("\nCombined continuous matrix shape:", X_continuous.shape)

X_binary = (X_continuous > 0).astype(np.int32)
print("Combined binary matrix shape:", X_binary.shape)

L1 subset shape: (5000, 30)
L2 subset shape: (5000, 46)

Combined continuous matrix shape: (5000, 76)
Combined binary matrix shape: (5000, 76)


In [7]:
import json

PREPARED_DIR = f"{PROJECT_DIR}/activations"

np.save(f"{PREPARED_DIR}/X_continuous.npy", X_continuous)
np.save(f"{PREPARED_DIR}/X_binary.npy", X_binary)
np.save(f"{PREPARED_DIR}/indices_L1.npy", indices_L1)
np.save(f"{PREPARED_DIR}/indices_L2.npy", indices_L2)

metadata = {
    "n_prompts": int(X_continuous.shape[0]),
    "n_features_L1": int(len(indices_L1)),
    "n_features_L2": int(len(indices_L2)),
    "n_features_total": int(X_continuous.shape[1]),
    "filter_lower": LOWER,
    "filter_upper": UPPER,
    "column_layout": "columns 0 to {} are L1 features, columns {} to {} are L2 features".format(
        len(indices_L1) - 1, len(indices_L1), X_continuous.shape[1] - 1
    ),
    "position_in_prompt": 39,
    "l1_hook": "blocks.1.hook_resid_pre",
    "l2_hook": "blocks.2.hook_resid_pre",
}

with open(f"{PREPARED_DIR}/metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("Saved files:")
for fname in ["X_continuous.npy", "X_binary.npy", "indices_L1.npy", "indices_L2.npy", "metadata.json"]:
    print(f"  {PREPARED_DIR}/{fname}")

print("\nMetadata:")
print(json.dumps(metadata, indent=2))

Saved files:
  /content/drive/MyDrive/pgm-final/activations/X_continuous.npy
  /content/drive/MyDrive/pgm-final/activations/X_binary.npy
  /content/drive/MyDrive/pgm-final/activations/indices_L1.npy
  /content/drive/MyDrive/pgm-final/activations/indices_L2.npy
  /content/drive/MyDrive/pgm-final/activations/metadata.json

Metadata:
{
  "n_prompts": 5000,
  "n_features_L1": 30,
  "n_features_L2": 46,
  "n_features_total": 76,
  "filter_lower": 0.05,
  "filter_upper": 0.95,
  "column_layout": "columns 0 to 29 are L1 features, columns 30 to 75 are L2 features",
  "position_in_prompt": 39,
  "l1_hook": "blocks.1.hook_resid_pre",
  "l2_hook": "blocks.2.hook_resid_pre"
}


In [8]:
X_cont_check = np.load(f"{PREPARED_DIR}/X_continuous.npy")
X_bin_check = np.load(f"{PREPARED_DIR}/X_binary.npy")
idx_L1_check = np.load(f"{PREPARED_DIR}/indices_L1.npy")
idx_L2_check = np.load(f"{PREPARED_DIR}/indices_L2.npy")

with open(f"{PREPARED_DIR}/metadata.json", "r") as f:
    meta_check = json.load(f)

print("Continuous matches:", np.array_equal(X_cont_check, X_continuous))
print("Binary matches:   ", np.array_equal(X_bin_check, X_binary))
print("L1 indices match: ", np.array_equal(idx_L1_check, indices_L1))
print("L2 indices match: ", np.array_equal(idx_L2_check, indices_L2))
print("Metadata matches: ", meta_check == metadata)

Continuous matches: True
Binary matches:    True
L1 indices match:  True
L2 indices match:  True
Metadata matches:  True
